# บทที่ 1: พื้นฐานของโครงข่ายประสาทเทียม (Neural Network Basics)

ใน Notebook นี้ เราจะสร้างองค์ประกอบพื้นฐานของโครงข่ายประสาทเทียมตั้งแต่ฟังก์ชันซิกมอยด์ เซลล์ประสาทเทียม (Artificial Neuron) เพอร์เซ็ปตรอน (Perceptron) ไปจนถึงการแก้ปัญหา XOR ด้วยเพอร์เซ็ปตรอนหลายชั้น (Multi-Layer Perceptron)

**ศัพท์ที่สำคัญในบทนี้:**- เวกเตอร์ (vector) — อาร์เรย์หนึ่งมิติ- เมทริกซ์ (matrix) — อาร์เรย์สองมิติ- ค่าน้ำหนัก (weight) — พารามิเตอร์ที่ปรับได้- ค่าไบแอส (bias) — ค่าเลื่อน- อินพุต (input) — ข้อมูลนำเข้า- เอาต์พุต (output) — ผลลัพธ์- ค่าสูญเสีย (loss) — วัดความคลาดเคลื่อน- เกรเดียนต์ (gradient) — ทิศทางการปรับ- ฟังก์ชันกระตุ้น (activation function) — ฟังก์ชันไม่เป็นเชิงเส้น- โครงข่ายประสาทเทียม (neural network) — โมเดลแมชชีนเลิร์นนิง

## 1. นำเข้าไลบรารีที่จำเป็น (Import Libraries)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# ติดตั้งฟอนต์ภาษาไทยสำหรับ Google Colab
import subprocess, glob
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-tlwg-garuda'], 
               capture_output=True)

# ลงทะเบียนฟอนต์โดยตรง
from matplotlib.font_manager import fontManager
for font_file in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fontManager.addfont(font_file)

# ตั้งค่า Seaborn theme และฟอนต์ภาษาไทย
sns.set_theme(style='whitegrid', font='Garuda')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
%config InlineBackend.figure_format = 'retina'

## 2. ฟังก์ชัน Sigmoid และกราฟ

ฟังก์ชันซิกมอยด์เป็นฟังก์ชันกระตุ้นที่แปลงค่าอินพุตให้อยู่ในช่วง (0, 1)

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

In [ ]:
def sigmoid(x):
    """ฟังก์ชัน Sigmoid"""
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    """อนุพันธ์ของฟังก์ชัน Sigmoid"""
    s = sigmoid(x)
    return s * (1 - s)

# สร้างข้อมูลสำหรับ plot
x = np.linspace(-10, 10, 100)
y_sig = sigmoid(x)
y_deriv = sigmoid_derivative(x)

# แสดงกราฟ
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(x, y_sig, 'b-', linewidth=2, label='Sigmoid')
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('x')
axes[0].set_ylabel('σ(x)')
axes[0].set_title('ฟังก์ชัน Sigmoid')
axes[0].legend()

axes[1].plot(x, y_deriv, 'r-', linewidth=2, label="Sigmoid'")
axes[1].set_xlabel('x')
axes[1].set_ylabel("σ'(x)")
axes[1].set_title('อนุพันธ์ของ Sigmoid')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. การคำนวณของเซลล์ประสาทเทียม (Artificial Neuron)

เซลล์ประสาทเทียมคำนวณผลลัพธ์จากสมการ:

$$y = \sigma(\sum_{i=1}^{n} w_i x_i + b)$$

In [ ]:
def artificial_neuron(x, weights, bias):
    """
    คำนวณผลลัพธ์ของเซลล์ประสาทเทียม

    พารามิเตอร์:
    - x: เวกเตอร์อินพุต
    - weights: เวกเตอร์ค่าน้ำหนัก
    - bias: ค่าไบแอส

    ค่าที่คืน:
    - output: ผลลัพธ์หลังผ่านฟังก์ชันซิกมอยด์
    """
    z = np.dot(weights, x) + bias
    return sigmoid(z)

# ตัวอย่างการคำนวณ
x = np.array([0.5, 0.3, 0.8])
weights = np.array([0.2, -0.1, 0.4])
bias = 0.1

output = artificial_neuron(x, weights, bias)
print(f"อินพุต: {x}")
print(f"ค่าน้ำหนัก: {weights}")
print(f"ค่าไบแอส: {bias}")
print(f"เอาต์พุต: {output:.4f}")

## 4. เพอร์เซ็ปตรอน (Perceptron) และกฎการเรียนรู้ของเพอร์เซ็ปตรอน (Perceptron Learning Rule)

เพอร์เซ็ปตรอนเป็นโครงข่ายประสาทเทียมที่ง่ายที่สุด ใช้สำหรับปัญหาการจำแนกประเภทเชิงเส้น (Linear Classification)

In [ ]:
class Perceptron:
    """
    ตัวจำแนกประเภทแบบเพอร์เซ็ปตรอน (Perceptron Classifier)
    """
    def __init__(self, n_features, learning_rate=0.1, n_epochs=100):
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.lr = learning_rate
        self.n_epochs = n_epochs

    def activation(self, z):
        """ฟังก์ชันขั้นบันได (Step Function)"""
        return 1 if z >= 0 else 0

    def predict(self, x):
        """ทำนายผลลัพธ์"""
        z = np.dot(self.weights, x) + self.bias
        return self.activation(z)

    def fit(self, X, y):
        """
        เรียนรู้จากข้อมูลด้วยกฎการเรียนรู้ของเพอร์เซ็ปตรอน (Perceptron Learning Rule):
        w_ใหม่ = w_เดิม + อัตราการเรียนรู้ * (คำตอบจริง - ค่าทำนาย) * x
        b_ใหม่ = b_เดิม + อัตราการเรียนรู้ * (คำตอบจริง - ค่าทำนาย)
        """
        for epoch in range(self.n_epochs):
            errors = 0
            for xi, target in zip(X, y):
                prediction = self.predict(xi)
                error = target - prediction

                if error != 0:
                    self.weights += self.lr * error * xi
                    self.bias += self.lr * error
                    errors += 1

            if errors == 0:
                print(f"ลู่เข้าที่รอบการฝึกที่ {epoch + 1}")
                break

        return self

## 5. ลอจิกเกต (Logic Gates): AND, OR, XOR

In [ ]:
# ข้อมูลสำหรับลอจิกเกต
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

# แอนเกต (AND)
y_and = np.array([0, 0, 0, 1])

# ออเกต (OR)
y_or = np.array([0, 1, 1, 1])

# เกต XOR
y_xor = np.array([0, 1, 1, 0])

print("=== แอนเกต (AND) ===")
perceptron_and = Perceptron(n_features=2, learning_rate=0.1, n_epochs=100)
perceptron_and.fit(X, y_and)
print(f"ค่าน้ำหนัก: {perceptron_and.weights}")
print(f"ค่าไบแอส: {perceptron_and.bias}")
for xi, yi in zip(X, y_and):
    print(f"อินพุต: {xi}, คำตอบจริง: {yi}, ค่าทำนาย: {perceptron_and.predict(xi)}")

print("\n=== ออเกต (OR) ===")
perceptron_or = Perceptron(n_features=2, learning_rate=0.1, n_epochs=100)
perceptron_or.fit(X, y_or)
print(f"ค่าน้ำหนัก: {perceptron_or.weights}")
print(f"ค่าไบแอส: {perceptron_or.bias}")
for xi, yi in zip(X, y_or):
    print(f"อินพุต: {xi}, คำตอบจริง: {yi}, ค่าทำนาย: {perceptron_or.predict(xi)}")

## 6. ปัญหา XOR - เพอร์เซ็ปตรอนเดี่ยวไม่สามารถแก้ได้

In [ ]:
print("=== เกต XOR (เพอร์เซ็ปตรอนเดี่ยว - จะไม่ลู่เข้า) ===")
perceptron_xor = Perceptron(n_features=2, learning_rate=0.1, n_epochs=100)
perceptron_xor.fit(X, y_xor)
print(f"ค่าน้ำหนัก: {perceptron_xor.weights}")
print(f"ค่าไบแอส: {perceptron_xor.bias}")
print("\nผลการทำนาย:")
for xi, yi in zip(X, y_xor):
    print(f"อินพุต: {xi}, คำตอบจริง: {yi}, ค่าทำนาย: {perceptron_xor.predict(xi)}")

print("\nสรุป: เพอร์เซ็ปตรอนเดี่ยวไม่สามารถแก้ปัญหา XOR ได้ เพราะ XOR ไม่แยกได้เชิงเส้น (not linearly separable)")

## 7. การแสดงภาพขอบเขตการตัดสินใจ (Decision Boundary)

In [ ]:
def plot_decision_boundary(perceptron, X, y, title, ax):
    """แสดงขอบเขตการตัดสินใจ"""
    # สร้าง grid
    x_min, x_max = -0.5, 1.5
    y_min, y_max = -0.5, 1.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                         np.linspace(y_min, y_max, 100))

    # ทำนายทุกจุดใน grid
    Z = np.array([perceptron.predict(np.array([x, y]))
                  for x, y in zip(xx.ravel(), yy.ravel())])
    Z = Z.reshape(xx.shape)

    # Plot
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=100, edgecolors='black')
    ax.set_xlabel('x₁')
    ax.set_ylabel('x₂')
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

plot_decision_boundary(perceptron_and, X, y_and, 'แอนเกต (AND)', axes[0])
plot_decision_boundary(perceptron_or, X, y_or, 'ออเกต (OR)', axes[1])
plot_decision_boundary(perceptron_xor, X, y_xor, 'เกต XOR (ล้มเหลว)', axes[2])

plt.tight_layout()
plt.show()

## 8. การแก้ปัญหา XOR ด้วยเพอร์เซ็ปตรอนหลายชั้น (Multi-Layer Perceptron: MLP)

In [ ]:
class MLP:
    """
    เพอร์เซ็ปตรอนหลายชั้น (Multi-Layer Perceptron) สำหรับแก้ปัญหา XOR
    ใช้ค่าสูญเสียกำลังสอง L = 1/2 (a2 - y)^2 ตามที่บทที่ 1 นิยามไว้
    """
    def __init__(self, n_input, n_hidden, n_output, learning_rate=0.5):
        # กำหนดค่าน้ำหนักเริ่มต้น
        self.W1 = np.random.randn(n_hidden, n_input) * 0.5
        self.b1 = np.zeros((n_hidden, 1))
        self.W2 = np.random.randn(n_output, n_hidden) * 0.5
        self.b2 = np.zeros((n_output, 1))
        self.lr = learning_rate

    def forward(self, x):
        """การส่งผ่านสัญญาณไปข้างหน้า"""
        self.x = x.reshape(-1, 1)
        self.z1 = np.dot(self.W1, self.x) + self.b1
        self.a1 = sigmoid(self.z1)
        self.z2 = np.dot(self.W2, self.a1) + self.b2
        self.a2 = sigmoid(self.z2)
        return self.a2

    def backward(self, y):
        """การแพร่กระจายย้อนกลับ"""
        y = y.reshape(-1, 1)
        m = 1  # ขนาดแบตช์

        # เกรเดียนต์ของชั้นเอาต์พุต: dL/dz2 = (a2 - y) * f'(z2) ตาม chain rule ของค่าสูญเสียกำลังสอง
        dz2 = (self.a2 - y) * sigmoid_derivative(self.z2)
        dW2 = np.dot(dz2, self.a1.T)
        db2 = dz2

        # เกรเดียนต์ของชั้นซ่อน
        dz1 = np.dot(self.W2.T, dz2) * sigmoid_derivative(self.z1)
        dW1 = np.dot(dz1, self.x.T)
        db1 = dz1

        # ปรับค่าน้ำหนัก
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def train(self, X, y, epochs=10000):
        """ฝึกโครงข่าย"""
        for epoch in range(epochs):
            for xi, yi in zip(X, y):
                self.forward(xi)
                self.backward(yi)

    def predict(self, x):
        """ทำนาย"""
        return self.forward(x)[0, 0]

# สร้างและฝึก MLP
# ตรึงค่าสุ่มตั้งต้นให้ผลการฝึกซ้ำได้ เพราะชั้นซ่อนสองเซลล์ตกหลุมค่าต่ำสุดเฉพาะที่ได้บ่อย
np.random.seed(42)
mlp = MLP(n_input=2, n_hidden=2, n_output=1, learning_rate=0.5)
mlp.train(X, y_xor, epochs=10000)

print("=== XOR กับ MLP ===")
print(f"W1:\n{mlp.W1}")
print(f"b1:\n{mlp.b1}")
print(f"W2:\n{mlp.W2}")
print(f"b2:\n{mlp.b2}")
print("\nผลการทำนาย:")
for xi, yi in zip(X, y_xor):
    pred = mlp.predict(xi)
    print(f"อินพุต: {xi}, คำตอบจริง: {yi}, ค่าทำนาย: {pred:.4f} → {1 if pred > 0.5 else 0}")

## 9. แบบฝึกหัดการคำนวณ

### แบบฝึกหัดที่ 1: คำนวณผลลัพธ์ของ Sigmoid
จงคำนวณค่าซิกมอยด์ของค่าต่อไปนี้:
- σ(0) = ?
- σ(1) = ?
- σ(-1) = ?
- σ(5) = ?

In [ ]:
# เขียนโค้ดคำนวณที่นี่
values = [0, 1, -1, 5]
for v in values:
    print(f"σ({v}) = {sigmoid(v):.4f}")

### แบบฝึกหัดที่ 2: คำนวณผลลัพธ์ของเซลล์ประสาทเทียม
ให้เซลล์ประสาทเทียมมีค่าดังนี้:
- อินพุต: x₁ = 0.5, x₂ = 0.8, x₃ = 0.2
- ค่าน้ำหนัก: w₁ = 0.3, w₂ = -0.2, w₃ = 0.5
- ค่าไบแอส: b = 0.1

จงคำนวณ:
1. ค่า z = Σwᵢxᵢ + b
2. ผลลัพธ์ y = σ(z)

In [ ]:
# เขียนโค้ดคำนวณที่นี่
x = np.array([0.5, 0.8, 0.2])
weights = np.array([0.3, -0.2, 0.5])
bias = 0.1

z = np.dot(weights, x) + bias
y = sigmoid(z)

print(f"z = {z:.4f}")
print(f"y = σ(z) = {y:.4f}")

### แบบฝึกหัดที่ 3: การเรียนรู้ของเพอร์เซ็ปตรอน (Perceptron Learning)
ให้เพอร์เซ็ปตรอนเริ่มต้นด้วย:
- w = [0, 0], b = 0
- อัตราการเรียนรู้ = 0.1

จงคำนวณค่า w และ b หลังจากเรียนรู้จากตัวอย่าง:
- อินพุต: [1, 1], คำตอบจริง: 1, ค่าทำนาย: 0

In [ ]:
# เขียนโค้ดคำนวณที่นี่
w = np.array([0.0, 0.0])
b = 0.0
lr = 0.1

x = np.array([1, 1])
target = 1
prediction = 0
error = target - prediction

w_new = w + lr * error * x
b_new = b + lr * error

print(f"ก่อนอัปเดต: w = {w}, b = {b}")
print(f"หลังอัปเดต: w = {w_new}, b = {b_new}")

### แบบฝึกหัดที่ 4: คำนวณจำนวนพารามิเตอร์
จงคำนวณจำนวนพารามิเตอร์ของโครงข่ายประสาทเทียมที่มี:
- ชั้นอินพุต: 4 เซลล์ประสาท
- ชั้นซ่อนที่ 1: 8 เซลล์ประสาท
- ชั้นซ่อนที่ 2: 4 เซลล์ประสาท
- ชั้นเอาต์พุต: 2 เซลล์ประสาท

In [ ]:
# เขียนโค้ดคำนวณที่นี่
def count_parameters(layers):
    """คำนวณจำนวนพารามิเตอร์"""
    total = 0
    for i in range(len(layers) - 1):
        weights = layers[i] * layers[i+1]
        biases = layers[i+1]
        total += weights + biases
        print(f"ชั้น {i} → {i+1}: ค่าน้ำหนัก = {weights}, ค่าไบแอส = {biases}")
    return total

layers = [4, 8, 4, 2]
total_params = count_parameters(layers)
print(f"\nพารามิเตอร์ทั้งหมด: {total_params}")

## บทสรุป

Notebook นี้แสดงให้เห็นว่า:
1. เพอร์เซ็ปตรอนเดี่ยวสามารถแก้ปัญหาการจำแนกประเภทเชิงเส้นได้ (AND, OR)
2. เพอร์เซ็ปตรอนเดี่ยวไม่สามารถแก้ปัญหา XOR ได้ เพราะ XOR ไม่แยกได้เชิงเส้น
3. เพอร์เซ็ปตรอนหลายชั้น (MLP) สามารถแก้ปัญหา XOR ได้

ผู้อ่านสามารถทดลองเปลี่ยนค่าพารามิเตอร์ต่างๆ เพื่อสังเกตผลลัพธ์ที่เปลี่ยนแปลงไป